# Prueba de tracking (go / no-go) en Colab

Corre el pipeline de YOLO sobre **un segmento** y te dice si el proyecto es viable con tu cámara
(panorámica 180° con encuadre automático) **antes** de gastar horas de GPU en el partido completo.

**Antes de empezar**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4 sirve).
2. Sube a tu Google Drive una carpeta `sports-analytics` con: `src/`, `config/`, `segments.csv` y el segmento que vas a probar en `videos/segments/` (por ejemplo `s03.mp4`).
3. Ejecuta las celdas en orden. Si la celda de instalación te pide reiniciar el entorno, reinicia y vuelve a correr **desde la celda 1**.

**Lo que obtienes:** un video de diagnóstico con las detecciones, las tablas del contrato (`data/tracking/*.parquet` en tu Drive) y métricas de calidad para decidir.

## 1. Configuración

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR   = "/content/drive/MyDrive/sports-analytics"   # carpeta del repo en tu Drive
SEGMENT_ID = "s03"        # segmento a probar (el que tenga ~2 min), como aparece en segments.csv
STRIDE     = 3            # se analiza 1 de cada 3 frames (~8 fps). Con cámara que se mueve, no subas de 4
MIN_KEYPOINTS = 6         # keypoints de la cancha mínimos para aceptar una homografía (ver celda 7)
MAX_SECONDS   = 30        # primera corrida: solo 30 s para medir tiempo. Pon None para el segmento completo
WITH_BALL     = True      # el balón es lo más lento (inferencia por recortes); pon False para una corrida rápida

## 2. Instalación

In [ ]:
# `sports` (Roboflow) se instala sin dependencias para controlar la versión de supervision:
# las versiones nuevas (>= 0.28) marcan ByteTrack como obsoleto y 0.31 lo elimina.
!pip install -q --no-deps git+https://github.com/roboflow/sports.git
!pip install -q "supervision==0.27.0" ultralytics gdown umap-learn sentencepiece protobuf

In [ ]:
import sys, os, time
sys.path.insert(0, f"{REPO_DIR}/src")

import numpy as np, pandas as pd, cv2, torch, supervision as sv
import matplotlib.pyplot as plt
from sportsan.config import load_config
from sportsan.timebase import load_segments
from sportsan.tracking.geometry import fit_homography
from sportsan.tracking.models import load_models
from sportsan.tracking.pipeline import collect_player_crops, run_segment

os.chdir(REPO_DIR)
cfg = load_config(f"{REPO_DIR}/config/project.yaml")
segments = load_segments(f"{REPO_DIR}/segments.csv").set_index("segment_id")
VIDEO = f"{REPO_DIR}/videos/segments/{SEGMENT_ID}.mp4"
SEG_START = float(segments.loc[SEGMENT_ID, "start_s"])
info = sv.VideoInfo.from_video_path(VIDEO)
print(f"Segmento {SEGMENT_ID}: {info.resolution_wh}, {info.fps:.1f} fps, {info.total_frames/info.fps:.0f} s, empieza en {SEG_START:.0f} s del video original")
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Pesos de los modelos
Son los mismos tres modelos que usa `roboflow/sports` (jugadoras/árbitros, balón y keypoints de la cancha). Se descargan de Google Drive con `gdown`.

In [ ]:
import gdown
W = "/content/weights"; os.makedirs(W, exist_ok=True)
files = {  # ids tomados de examples/soccer/setup.sh de roboflow/sports
    "football-ball-detection.pt":   "1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V",
    "football-player-detection.pt": "17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q",
    "football-pitch-detection.pt":  "1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf",
}
for name, gid in files.items():
    if not os.path.exists(f"{W}/{name}"):
        gdown.download(f"https://drive.google.com/uc?id={gid}", f"{W}/{name}", quiet=False)
print(os.listdir(W))

In [ ]:
models = load_models(W, DEVICE, with_ball=WITH_BALL, with_teams=True)
print("Modelos listos")

## 4. Chequeo rápido de calibración (el primer indicador go / no-go)
Toma 12 frames repartidos por el segmento y mira **cuántos keypoints de la cancha ve el modelo** y si la homografía sale.
Con la cámara de seguimiento automático es normal ver pocos keypoints; lo que importa es qué porcentaje de frames alcanza el mínimo.

In [ ]:
def sample_frames(path, n=12):
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
        if ok: out.append((int(i), f))
    cap.release(); return out

W_, H_ = info.resolution_wh
rows, shown = [], []
for i, f in sample_frames(VIDEO, 12):
    xy, conf = models.detect_pitch(f)
    cal = fit_homography(xy, models.pitch_vertices_m, conf, min_points=MIN_KEYPOINTS, image_wh=(W_, H_))
    rows.append(dict(t_s=round(i / info.fps, 1), keypoints=cal.n_keypoints, homografia=cal.H is not None, motivo=cal.reason))
    img = f.copy()
    for (x, y), c in zip(xy, conf):
        if c >= 0.5 and x > 1 and y > 1: cv2.circle(img, (int(x), int(y)), 8, (0, 0, 255), -1)
    shown.append(img[..., ::-1])
print(pd.DataFrame(rows).to_string(index=False))
fig, axs = plt.subplots(3, 4, figsize=(20, 9))
for ax, im, r in zip(axs.ravel(), shown, rows):
    ax.imshow(im); ax.set_title(f"t={r['t_s']}s  kp={r['keypoints']}  H={'ok' if r['homografia'] else 'NO'}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Clasificación de equipos
Se agrupan las jugadoras en 2 clusters por apariencia. **Los números 0 y 1 son arbitrarios**: mira los recortes y
dime cuál es tu equipo en la celda siguiente. (Porteras y árbitros se resuelven solos.)

In [ ]:
t0 = time.time()
crops = collect_player_crops(VIDEO, models, fit_stride_frames=60)
print(f"{len(crops)} recortes en {time.time()-t0:.0f} s")
models.team_fit(crops)
rng = np.random.default_rng(0)
sample = [crops[i] for i in rng.choice(len(crops), min(len(crops), 200), replace=False)]
clusters = np.asarray(models.team_predict(sample))
fig, axs = plt.subplots(2, 1, figsize=(16, 6))
for c in (0, 1):
    picks = [s for s, k in zip(sample, clusters) if k == c][:12]
    strip = np.hstack([cv2.resize(p, (64, 128)) for p in picks]) if picks else np.zeros((128, 64, 3), np.uint8)
    axs[c].imshow(strip[..., ::-1]); axs[c].set_title(f"Cluster {c}  ({int((clusters == c).sum())} de {len(sample)} recortes)"); axs[c].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
OWN_CLUSTER = 0   # <-- 0 o 1: el cluster que muestra a TU equipo

## 6. Correr el segmento

In [ ]:
max_frames = int(MAX_SECONDS * info.fps / STRIDE) if MAX_SECONDS else None
os.makedirs(f"{REPO_DIR}/data/preview", exist_ok=True)
tracking, ball, calib, stats = run_segment(
    VIDEO, SEGMENT_ID, SEG_START, models, match_id=cfg["match_id"], own_cluster=OWN_CLUSTER,
    stride=STRIDE, min_keypoints=MIN_KEYPOINTS, max_frames=max_frames,
    preview_path=f"{REPO_DIR}/data/preview/{SEGMENT_ID}_raw.mp4",
    out_dir=f"{REPO_DIR}/data/tracking")
for k, v in stats.items(): print(f"{k:45s} {v}")
hours = stats["seg_computo_por_seg_de_video"] * 90 * 60 / 3600
print(f"\nEstimado para 90 min de juego a este ritmo: ~{hours:.1f} h de GPU (repartible entre sesiones/cuentas)")

## 7. Video de diagnóstico
Azul = tu equipo, rojo = rival, gris = árbitro, amarillo = sin equipo. El número es el ID de tracking. Abajo, mini-radar con las posiciones en la cancha.

In [ ]:
src, dst = f"{REPO_DIR}/data/preview/{SEGMENT_ID}_raw.mp4", f"/content/{SEGMENT_ID}_preview.mp4"
!ffmpeg -y -loglevel error -i "{src}" -vf scale=960:-2 -c:v libx264 -crf 28 -pix_fmt yuv420p "{dst}"
from IPython.display import Video
Video(dst, embed=True, width=900)

## 8. Diagnóstico

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 9))

ax = axs[0, 0]
colors = np.where(calib["ok"], "tab:green", "tab:red")
ax.scatter(calib["t_video"], calib["n_keypoints"], c=colors, s=10)
ax.axhline(MIN_KEYPOINTS, ls="--", c="gray"); ax.set_title("Keypoints visibles por frame (verde = homografía aceptada)")
ax.set_xlabel("tiempo del video original (s)")

ax = axs[0, 1]
print("% de frames con al menos N keypoints (antes de otros filtros):")
for k in (4, 6, 8, 10):
    print(f"   N={k:2d}:  {100*(calib['n_keypoints'] >= k).mean():5.1f} %")
print("\nMotivos de rechazo:"); print(calib["reason"].value_counts().to_string())
calib["reason"].value_counts().plot.bar(ax=ax, color="tab:orange"); ax.set_title("Resultado de la calibración")

ax = axs[1, 0]
dur = tracking.groupby("track_id")["frame"].nunique() * STRIDE / info.fps
ax.hist(dur, bins=30); ax.set_title(f"Duración de cada track (s)  mediana={dur.median():.1f}  tracks={len(dur)}")

ax = axs[1, 1]
for team, c in (("own", "tab:blue"), ("opp", "tab:red")):
    d = tracking[tracking["team"] == team].dropna(subset=["x", "y"])
    ax.scatter(d["x"], d["y"], s=3, alpha=0.25, c=c, label=f"{team} ({len(d)})")
ax.set_xlim(0, 105); ax.set_ylim(68, 0); ax.set_aspect("equal"); ax.legend(); ax.set_title("Posiciones en la cancha (m)")
plt.tight_layout(); plt.show()

## 9. ¿Go o no-go?
Reglas de dedo, **no son umbrales validados**; sirven para decidir qué corregir. Mira el video de la celda 7 antes que los números.

| Indicador | Bien | Revisar |
|---|---|---|
| `pct_homografia_ok` | > ~60 % del juego | < ~40 %: bajar `MIN_KEYPOINTS` a 4–5 (más frames, más error) o probar otro modelo de keypoints |
| `duracion_track_mediana_s` | varios segundos | pocos segundos con muchos tracks: hay cambios de ID (¿`STRIDE` muy alto?) |
| Equipos | los 2 clusters separan bien las camisetas | mezclados: añadir más recortes o revisar iluminación |
| Posiciones en la cancha | jugadoras dentro de la cancha, por equipo coherentes con el video | puntos "volando" fuera de zona o espejados (orientación invertida) |
| `pct_frames_con_balon` | > ~50 % | bajo es normal en 1080p con balón pequeño: se compensa con el etiquetado manual |

**Recuerda:** esta cámara solo muestra el encuadre que sigue el juego. Las jugadoras fuera de cuadro no se ven, así que ancho/largo del equipo
y mapas de calor tendrán huecos (ver `docs/data_contract.md`).

Cuando termines, **descarga a tu laptop** `data/tracking/*_{SEGMENT_ID}.parquet` y `data/preview/` desde Drive, y cuéntame: el bloque de `stats`,
cuántos keypoints se ven en las 12 muestras y qué viste en el video.